# 02 数据质量审计与清洗

本文件只做数据质量审计和清洗，不做风险表现分析。

职责边界：

- `02`：识别数据问题、定义处理口径、生成清洗后数据；
- `03`：基于清洗后数据计算坏客户率、风险倍数和高风险分组。


## 1. 本阶段产出

本阶段产出两个文件：

1. `credit_risk_cleaned.csv`：后续风险因子分析使用的主数据；
2. `cleaning_audit_summary.csv`：清洗审计记录，用于解释每类问题如何处理。

本阶段不判断哪个变量风险更高，只保证后续分析口径准确。


## 2. 导入工具、设置路径、读取数据


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

PROJECT_ROOT = Path(r"E:\DataAnalysis\DataAnalysisRoad\Projects\01_Credit_Risk_Analysis")
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cs-training.csv"

RAW_DATA_PATH


WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/data/raw/cs-training.csv')

In [2]:
raw_df = pd.read_csv(RAW_DATA_PATH)

rename_cols = {
    "id": "customer_id",
    "SeriousDlqin2yrs": "is_bad_customer",
    "RevolvingUtilizationOfUnsecuredLines": "credit_utilization",
    "age": "age",
    "NumberOfTime30-59DaysPastDueNotWorse": "past_due_30_59_count",
    "DebtRatio": "debt_ratio",
    "MonthlyIncome": "monthly_income",
    "NumberOfOpenCreditLinesAndLoans": "open_credit_count",
    "NumberOfTimes90DaysLate": "past_due_90_count",
    "NumberRealEstateLoansOrLines": "real_estate_loan_count",
    "NumberOfTime60-89DaysPastDueNotWorse": "past_due_60_89_count",
    "NumberOfDependents": "dependent_count",
}

df = raw_df.rename(columns=rename_cols)
df = df.drop(columns=["timestamp"])

sample_count = len(df)

df.head()


,customer_id,is_bad_customer,credit_utilization,age,past_due_30_59_count,debt_ratio,monthly_income,open_credit_count,past_due_90_count,real_estate_loan_count,past_due_60_89_count,dependent_count
0,1,1,0.7661,45,2,0.8030,"9,120.0000",13,0,6,0,2.0000
1,2,0,0.9572,40,0,0.1219,"2,600.0000",4,0,0,0,1.0000
2,3,0,0.6582,38,1,0.0851,"3,042.0000",2,1,0,0,0.0000
3,4,0,0.2338,30,0,0.0360,"3,300.0000",5,0,0,0,0.0000
4,5,0,0.9072,49,1,0.0249,"63,588.0000",7,0,1,0,0.0000


## 3. 统一定义审计条件

这一节只定义条件，不做分析、不做清洗。

后续所有审计表和清洗规则都引用这里的条件，避免口径分散。


In [4]:
# 缺失值条件
income_missing = df["monthly_income"].isna()
dependent_missing = df["dependent_count"].isna()

# 逾期特殊编码条件：96/98 不作为真实逾期次数使用
past_due_30_special = df["past_due_30_59_count"].isin([96, 98])
past_due_60_special = df["past_due_60_89_count"].isin([96, 98])
past_due_90_special = df["past_due_90_count"].isin([96, 98])
past_due_special = past_due_30_special | past_due_60_special | past_due_90_special

# 明确可判断逾期状态的客户：排除逾期字段特殊编码客户
past_due_normal = ~past_due_special

# 极端值条件
age_zero = df["age"].eq(0)
age_over_100 = df["age"].gt(100)
util_over_1 = df["credit_utilization"].gt(1)
util_over_10 = df["credit_utilization"].gt(10)
open_credit_over_40 = df["open_credit_count"].gt(40)
dependent_over_10 = df["dependent_count"].gt(10)


## 4. 清洗审计表

这张表只记录数据问题、影响范围和处理方式。


In [5]:
cleaning_audit_summary = pd.DataFrame({
    "检查项": [
        "月收入缺失",
        "家属数量缺失",
        "年龄为0",
        "年龄大于100",
        "额度使用率大于1",
        "额度使用率大于10",
        "开户/贷款账户数大于40",
        "家属数量大于10",
        "30-59天逾期为96/98",
        "60-89天逾期为96/98",
        "90天以上逾期为96/98",
        "任一逾期字段为96/98",
    ],
    "人数": [
        income_missing.sum(),
        dependent_missing.sum(),
        age_zero.sum(),
        age_over_100.sum(),
        util_over_1.sum(),
        util_over_10.sum(),
        open_credit_over_40.sum(),
        dependent_over_10.sum(),
        past_due_30_special.sum(),
        past_due_60_special.sum(),
        past_due_90_special.sum(),
        past_due_special.sum(),
    ],
    "占比": [
        income_missing.mean(),
        dependent_missing.mean(),
        age_zero.mean(),
        age_over_100.mean(),
        util_over_1.mean(),
        util_over_10.mean(),
        open_credit_over_40.mean(),
        dependent_over_10.mean(),
        past_due_30_special.mean(),
        past_due_60_special.mean(),
        past_due_90_special.mean(),
        past_due_special.mean(),
    ],
    "处理方式": [
        "保留原值，增加缺失标记",
        "保留原值，增加缺失标记",
        "置为空，增加异常标记",
        "保留原值，增加极端值标记",
        "保留原值，增加极端值标记",
        "保留原值，后续分组观察",
        "保留原值，增加极端值标记",
        "保留原值，增加极端值标记",
        "置为空，增加特殊编码标记",
        "置为空，增加特殊编码标记",
        "置为空，增加特殊编码标记",
        "DPD30+ 状态设为无法判断",
    ],
    "处理原因": [
        "缺失可能包含业务含义，不能直接删除",
        "缺失可能包含业务含义，不能直接删除",
        "0岁不符合业务常识",
        "极端但可能真实，不直接删除",
        "可能代表超额使用或数据异常，不直接删除",
        "极端值较高，后续在分组分析中观察",
        "可能代表多头借贷或特殊客户，不直接删除",
        "样本极少，保留观察",
        "疑似特殊编码，不可当作真实逾期次数",
        "疑似特殊编码，不可当作真实逾期次数",
        "疑似特殊编码，不可当作真实逾期次数",
        "避免特殊编码客户被误归为无逾期",
    ],
})

cleaning_audit_summary


,检查项,人数,占比,处理方式,处理原因
0,月收入缺失,29731,0.1982,保留原值，增加缺失标记,缺失可能包含业务含义，不能直接删除
1,家属数量缺失,3924,0.0262,保留原值，增加缺失标记,缺失可能包含业务含义，不能直接删除
2,年龄为0,1,0.0000,置为空，增加异常标记,0岁不符合业务常识
3,年龄大于100,13,0.0001,保留原值，增加极端值标记,极端但可能真实，不直接删除
4,额度使用率大于1,3321,0.0221,保留原值，增加极端值标记,可能代表超额使用或数据异常，不直接删除
5,额度使用率大于10,241,0.0016,保留原值，后续分组观察,极端值较高，后续在分组分析中观察
6,开户/贷款账户数大于40,62,0.0004,保留原值，增加极端值标记,可能代表多头借贷或特殊客户，不直接删除
7,家属数量大于10,2,0.0000,保留原值，增加极端值标记,样本极少，保留观察
8,30-59天逾期为96/98,269,0.0018,置为空，增加特殊编码标记,疑似特殊编码，不可当作真实逾期次数
9,60-89天逾期为96/98,269,0.0018,置为空，增加特殊编码标记,疑似特殊编码，不可当作真实逾期次数


## 5. 执行清洗规则

这一节真正生成 `clean_df`。

核心口径：

- 明显错误值置为空；
- 96/98 特殊编码置为空；
- 缺失、异常、特殊编码都保留标记；
- 特殊编码客户的 `ever_dpd30_plus` 设为 `NaN`，表示无法判断。


In [8]:
clean_df = df.copy()

# 1. 缺失值标记
clean_df["is_monthly_income_missing"] = income_missing.astype(int)
clean_df["is_dependent_count_missing"] = dependent_missing.astype(int)

# 2. 年龄为 0：置为空，并保留标记
clean_df["is_age_zero"] = age_zero.astype(int)
clean_df.loc[age_zero, "age"] = np.nan

# 3. 极端值标记
clean_df["is_age_over_100"] = age_over_100.astype(int)
clean_df["is_credit_utilization_over_1"] = util_over_1.astype(int)
clean_df["is_open_credit_over_40"] = open_credit_over_40.astype(int)
clean_df["is_dependent_over_10"] = dependent_over_10.astype(int)

# 4. 逾期特殊编码：先保留标记，再置为空
clean_df["is_past_due_30_59_special"] = past_due_30_special.astype(int)
clean_df["is_past_due_60_89_special"] = past_due_60_special.astype(int)
clean_df["is_past_due_90_special"] = past_due_90_special.astype(int)

clean_df.loc[past_due_30_special, "past_due_30_59_count"] = np.nan
clean_df.loc[past_due_60_special, "past_due_60_89_count"] = np.nan
clean_df.loc[past_due_90_special, "past_due_90_count"] = np.nan

# 5. DPD30+ 派生变量
# 1 = 明确有 DPD30+
# 0 = 明确无 DPD30+
# NaN = 逾期字段为特殊编码，无法判断
clean_df["ever_dpd30_plus"] = np.nan

clean_df.loc[past_due_normal, "ever_dpd30_plus"] = (
    clean_df.loc[past_due_normal, "past_due_30_59_count"].gt(0)
    | clean_df.loc[past_due_normal, "past_due_60_89_count"].gt(0)
    | clean_df.loc[past_due_normal, "past_due_90_count"].gt(0)
).astype(int)



clean_df.head()


,customer_id,is_bad_customer,credit_utilization,age,past_due_30_59_count,debt_ratio,monthly_income,open_credit_count,past_due_90_count,real_estate_loan_count,past_due_60_89_count,dependent_count,is_monthly_income_missing,is_dependent_count_missing,is_age_zero,is_age_over_100,is_credit_utilization_over_1,is_open_credit_over_40,is_dependent_over_10,is_past_due_30_59_special,is_past_due_60_89_special,is_past_due_90_special,ever_dpd30_plus
0,1,1,0.7661,45.0000,2.0000,0.8030,"9,120.0000",13,0.0000,6,0.0000,2.0000,0,0,0,0,0,0,0,0,0,0,1.0000
1,2,0,0.9572,40.0000,0.0000,0.1219,"2,600.0000",4,0.0000,0,0.0000,1.0000,0,0,0,0,0,0,0,0,0,0,0.0000
2,3,0,0.6582,38.0000,1.0000,0.0851,"3,042.0000",2,1.0000,0,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,1.0000
3,4,0,0.2338,30.0000,0.0000,0.0360,"3,300.0000",5,0.0000,0,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,0.0000
4,5,0,0.9072,49.0000,1.0000,0.0249,"63,588.0000",7,0.0000,1,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,1.0000


## 6. 清洗结果校验

校验只看清洗是否按规则执行，不做风险结论。


In [9]:
past_due_special_after_cleaning = (
    clean_df["past_due_30_59_count"].isin([96, 98])
    | clean_df["past_due_60_89_count"].isin([96, 98])
    | clean_df["past_due_90_count"].isin([96, 98])
)

cleaning_check = pd.DataFrame({
    "检查项": [
        "清洗前样本数",
        "清洗后样本数",
        "清洗前年龄为0",
        "清洗后年龄为0",
        "清洗前逾期字段含96/98",
        "清洗后逾期字段含96/98",
        "ever_dpd30_plus 无法判断人数",
    ],
    "结果": [
        len(df),
        len(clean_df),
        age_zero.sum(),
        clean_df["age"].eq(0).sum(),
        past_due_special.sum(),
        past_due_special_after_cleaning.sum(),
        clean_df["ever_dpd30_plus"].isna().sum(),
    ],
})

cleaning_check


,检查项,结果
0,清洗前样本数,150000
1,清洗后样本数,150000
2,清洗前年龄为0,1
3,清洗后年龄为0,0
4,清洗前逾期字段含96/98,269
5,清洗后逾期字段含96/98,0
6,ever_dpd30_plus 无法判断人数,269


In [10]:
dpd30_status_count = clean_df["ever_dpd30_plus"].value_counts(dropna=False).reset_index()
dpd30_status_count.columns = ["ever_dpd30_plus", "人数"]
dpd30_status_count["占比"] = dpd30_status_count["人数"] / len(clean_df)

dpd30_status_count


,ever_dpd30_plus,人数,占比
0,0.0000,119637,0.7976
1,1.0000,30094,0.2006
2,NaN,269,0.0018


## 7. 保存清洗结果


In [ ]:
processed_dir = PROJECT_ROOT / "data" / "processed"
output_dir = PROJECT_ROOT / "output"

processed_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

clean_data_path = processed_dir / "credit_risk_cleaned.csv"
audit_summary_path = output_dir / "cleaning_audit_summary.csv"

clean_df.to_csv(clean_data_path, index=False, encoding="utf-8-sig")
cleaning_audit_summary.to_csv(audit_summary_path, index=False, encoding="utf-8-sig")

clean_data_path, audit_summary_path


(WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/data/processed/credit_risk_cleaned.csv'),
 WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/output/cleaning_audit_summary.csv'))

: 

## 8. 本文件结论

- `02` 只完成数据清洗，不输出风险强弱结论；
- 清洗后样本数保持不变；
- 年龄为 0 已置为空；
- 逾期字段 96/98 已置为空，并保留特殊编码标记；
- `ever_dpd30_plus` 保留三种状态：有、无、无法判断；
- 下一步进入 `03`，用清洗后的数据正式分析坏客户率和风险因子。
